# Synthetic Data Generation with CopulaGAN Synthesizer using SDV

This notebook demonstrates how to generate synthetic tabular data and train a model using the `CopulaGANSynthesizer` from the Synthetic Data Vault (SDV) library.

We will use a generated dataset of shape **(100, 50)** (100 rows and 50 columns) containing a mix of numerical, categorical, and boolean variables.

In [ ]:
# Install the SDV library
!pip install sdv

## 1. Import Libraries
We import pandas, numpy, and the necessary modules from SDV.

In [ ]:
import pandas as pd
import numpy as np
from sdv.metadata import SingleTableMetadata
from sdv.single_table import CopulaGANSynthesizer
from sdv.evaluation.single_table import evaluate_quality, run_diagnostic

print("Libraries imported successfully!")

## 2. Generate a Mock Dataset (100 rows, 50 columns)
We will programmatically construct a dataset with 100 rows and 50 columns containing:
- 1 Unique Primary ID column (`id`)
- 30 Numerical columns with varying distributions (`num_col_0` to `num_col_29`)
- 10 Categorical columns with varying categories (`cat_col_0` to `cat_col_9`)
- 9 Boolean columns (`bool_col_0` to `bool_col_8`)

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

num_rows = 100
data = {}

# 1. Primary ID Column (1 column)
data['id'] = [f'ID_{i:03d}' for i in range(num_rows)]

# 2. Numerical Columns (30 columns)
for i in range(30):
    col_name = f'num_col_{i}'
    dist_type = i % 4
    if dist_type == 0:
        data[col_name] = np.random.normal(loc=50.0, scale=15.0, size=num_rows)
    elif dist_type == 1:
        data[col_name] = np.random.uniform(low=10.0, high=200.0, size=num_rows)
    elif dist_type == 2:
        data[col_name] = np.random.exponential(scale=20.0, size=num_rows)
    else:
        data[col_name] = np.random.randint(low=1, high=1000, size=num_rows)

# 3. Categorical Columns (10 columns)
for i in range(10):
    col_name = f'cat_col_{i}'
    categories = [f'Cat_{chr(65 + j)}' for j in range(3 + (i % 3))] # 3 to 5 categories
    data[col_name] = np.random.choice(categories, size=num_rows)

# 4. Boolean Columns (9 columns)
for i in range(9):
    col_name = f'bool_col_{i}'
    data[col_name] = np.random.choice([True, False], size=num_rows)

# Create DataFrame
real_data = pd.DataFrame(data)

# Verify shape (100, 50)
print(f"Dataset shape: {real_data.shape}")
real_data.head()

## 3. Define and Detect Metadata
SDV requires a metadata schema defining the role of each column (e.g., numerical, categorical, ID). We will use `SingleTableMetadata` to automatically detect column types, then customize the primary key.

In [ ]:
# Initialize metadata and detect columns automatically
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data=real_data)

# Set the ID column as the primary key
metadata.set_primary_key(column_name='id')

# Visualize the metadata structure
metadata_dict = metadata.to_dict()
print(f"Primary key: {metadata_dict.get('primary_key')}")
print("Columns detected:")
for col, details in metadata_dict['columns'].items():
    print(f"  - {col}: {details['sdtype']}")

## 4. Train the CopulaGAN Synthesizer
`CopulaGANSynthesizer` blends classical statistical methods (copulas) with Generative Adversarial Networks (GANs). We initialize the synthesizer and fit it to our real dataset.

In [ ]:
# Initialize the synthesizer
synthesizer = CopulaGANSynthesizer(metadata)

# Train the synthesizer on our real data
print("Training the CopulaGAN model...")
synthesizer.fit(real_data)
print("Training completed!")

## 5. Generate Synthetic Data
With the model trained, we can now sample synthetic rows. Let's generate 100 new records.

In [ ]:
# Sample 100 rows of synthetic data
synthetic_data = synthesizer.sample(num_rows=100)

print(f"Synthetic data shape: {synthetic_data.shape}")
synthetic_data.head()

## 6. Evaluate the Quality of Synthetic Data
SDV provides evaluation functions to compare how similar the synthetic data is to the real data. We will compute a quality report and run diagnostics.

In [ ]:
# 1. Run quality evaluation
print("Evaluating quality...")
quality_report = evaluate_quality(
    real_data=real_data,
    synthetic_data=synthetic_data,
    metadata=metadata
)

# 2. Run diagnostic checks
print("\nRunning diagnostics...")
diagnostic_report = run_diagnostic(
    real_data=real_data,
    synthetic_data=synthetic_data,
    metadata=metadata
)

## 7. Saving and Loading the Model
We can save the trained synthesizer to a file for later use and load it back.

In [ ]:
# Save the synthesizer model
synthesizer.save('copulagan_synthesizer.pkl')
print("Model saved to 'copulagan_synthesizer.pkl'")

# Load the synthesizer model back
loaded_synthesizer = CopulaGANSynthesizer.load('copulagan_synthesizer.pkl')
new_synthetic_data = loaded_synthesizer.sample(num_rows=10)
print(f"Successfully loaded model and generated {len(new_synthetic_data)} rows!")